# Notebook 01 — Data Collection
## China National Sword Natural Experiment: U.S. Recycling Export Flows

**Project:** Did China's January 2018 National Sword policy measurably redirect U.S. recyclable material exports?

**This notebook:**
- Pulls U.S. scrap export data from UN Comtrade (2015–2023)
- Collects for key recyclable commodity codes: paper, plastics, ferrous metals, copper, aluminium
- Two partner slices per year: China (156) and World (0)
- Saves raw data to `data/raw/comtrade_raw.csv`

**Data source:** [UN Comtrade Public Preview API](https://comtradeapi.un.org)  
**No API key required** for the public preview endpoint (rate-limited to ~100 requests/hour)

---
### HS Commodity Codes
| Code | Description |
|------|-------------|
| 4707 | Recovered (waste & scrap) paper / paperboard |
| 3915 | Waste, parings & scrap of plastics |
| 7204 | Ferrous waste & scrap (iron/steel) |
| 7404 | Copper waste & scrap |
| 7602 | Aluminium waste & scrap |

### Key Parameters
- **Reporter:** 842 (United States)
- **Flow:** X (Exports)
- **Partners:** 156 (China), 0 (World total)
- **Frequency:** Annual
- **Years:** 2015–2023

In [ ]:
import requests
import pandas as pd
import time
import os
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = RAW_DIR / 'comtrade_raw.csv'

print('Directories ready.')
print(f'Output path: {OUT_FILE}')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────

BASE_URL = 'https://comtradeapi.un.org/public/v1/preview/C/A/HS'

REPORTER  = 842          # United States
FLOW      = 'X'          # Exports
YEARS     = list(range(2015, 2024))   # 2015–2023

# Partners: China + World (for share calculations)
PARTNERS = {
    156: 'China',
    0:   'World'
}

# Commodity codes: HS waste/scrap categories most affected by National Sword
COMMODITIES = {
    '4707': 'Waste Paper / Paperboard',
    '3915': 'Plastic Waste & Scrap',
    '7204': 'Ferrous Waste & Scrap',
    '7404': 'Copper Waste & Scrap',
    '7602': 'Aluminium Waste & Scrap',
}

print(f'Years to collect: {YEARS}')
print(f'Commodities: {list(COMMODITIES.keys())}')
print(f'Partners: {PARTNERS}')
total_calls = len(YEARS) * len(PARTNERS) * len(COMMODITIES)
print(f'\nTotal API calls needed: {total_calls}')
print(f'Estimated time at 1s delay: ~{total_calls}s')

In [ ]:
# ── Fetch function ─────────────────────────────────────────────────────────

def fetch_comtrade(year, partner_code, cmd_code, delay=1.1):
    """
    Fetch a single year/partner/commodity record from UN Comtrade public API.
    Returns a dict with the key fields, or None on failure.
    """
    params = {
        'reporterCode': REPORTER,
        'partnerCode':  partner_code,
        'cmdCode':      cmd_code,
        'flowCode':     FLOW,
        'period':       year,
    }
    try:
        resp = requests.get(BASE_URL, params=params, timeout=15)
        resp.raise_for_status()
        result = resp.json()

        if result.get('error'):
            print(f'  API error [{year}/{partner_code}/{cmd_code}]: {result["error"]}')
            return None

        data = result.get('data', [])
        if not data:
            # No trade reported for this combination — valid zero
            return {
                'year':          year,
                'partner_code':  partner_code,
                'partner_name':  PARTNERS.get(partner_code, str(partner_code)),
                'cmd_code':      cmd_code,
                'cmd_name':      COMMODITIES.get(str(cmd_code), str(cmd_code)),
                'net_weight_kg': 0.0,
                'fob_value_usd': 0.0,
                'is_reported':   False,
            }

        row = data[0]  # Annual aggregate — always one row
        return {
            'year':          year,
            'partner_code':  partner_code,
            'partner_name':  PARTNERS.get(partner_code, str(partner_code)),
            'cmd_code':      cmd_code,
            'cmd_name':      COMMODITIES.get(str(cmd_code), str(cmd_code)),
            'net_weight_kg': row.get('netWgt', 0.0),
            'fob_value_usd': row.get('primaryValue', 0.0),
            'is_reported':   row.get('isReported', False),
        }

    except requests.exceptions.RequestException as e:
        print(f'  Request failed [{year}/{partner_code}/{cmd_code}]: {e}')
        return None
    finally:
        time.sleep(delay)  # Respect rate limit

print('fetch_comtrade() defined.')

In [ ]:
# ── Main collection loop ───────────────────────────────────────────────────
#
# Loops: years × partners × commodities
# Saves progress every 10 records in case of interruption.

records = []
errors  = []
call_n  = 0

for year in YEARS:
    for partner_code, partner_name in PARTNERS.items():
        for cmd_code, cmd_name in COMMODITIES.items():
            call_n += 1
            print(f'[{call_n}/{total_calls}] {year} | {partner_name} | {cmd_name}...', end=' ')

            row = fetch_comtrade(year, partner_code, cmd_code)

            if row:
                records.append(row)
                weight_mt = row['net_weight_kg'] / 1e6
                value_m   = row['fob_value_usd'] / 1e6
                print(f'{weight_mt:,.0f} MT | ${value_m:,.1f}M')
            else:
                errors.append((year, partner_code, cmd_code))
                print('FAILED')

print(f'\n✓ Done. {len(records)} records collected, {len(errors)} errors.')

In [ ]:
# ── Save raw data ──────────────────────────────────────────────────────────

df_raw = pd.DataFrame(records)

# Convert weight to metric tons for readability
df_raw['net_weight_mt'] = df_raw['net_weight_kg'] / 1e6

# Reorder columns cleanly
df_raw = df_raw[[
    'year', 'partner_code', 'partner_name',
    'cmd_code', 'cmd_name',
    'net_weight_kg', 'net_weight_mt', 'fob_value_usd', 'is_reported'
]]

df_raw.to_csv(OUT_FILE, index=False)
print(f'Saved {len(df_raw)} rows → {OUT_FILE}')
print(f'\nShape: {df_raw.shape}')
print(f'Years: {sorted(df_raw.year.unique())}')
print(f'Partners: {df_raw.partner_name.unique()}')
print(f'Commodities: {df_raw.cmd_code.unique()}')

In [ ]:
# ── Quick sanity check ─────────────────────────────────────────────────────
# Spot-check the National Sword cliff: waste paper exports to China

paper_china = df_raw[
    (df_raw['cmd_code'] == '4707') &
    (df_raw['partner_name'] == 'China')
][['year', 'net_weight_mt', 'fob_value_usd']].sort_values('year')

paper_china['value_M'] = paper_china['fob_value_usd'] / 1e6

print('Waste Paper (HS 4707) — U.S. Exports to China')
print('=' * 50)
for _, r in paper_china.iterrows():
    bar = '█' * int(r['net_weight_mt'] / 500)
    marker = ' ← NATIONAL SWORD' if r['year'] == 2018 else ''
    print(f"{int(r['year'])} | {r['net_weight_mt']:>8,.0f} MT | ${r['value_M']:>6,.0f}M  {bar}{marker}")

if len(errors) > 0:
    print(f'\n⚠ Failed calls: {errors}')

---
## Data Collection Complete

**Output:** `data/raw/comtrade_raw.csv`

**Columns:**
- `year` — Reference year (2015–2023)
- `partner_code` — UN country code (156=China, 0=World)
- `partner_name` — Human-readable partner label
- `cmd_code` — HS commodity code
- `cmd_name` — Commodity description
- `net_weight_kg` — Net weight in kilograms
- `net_weight_mt` — Net weight in metric tons (derived)
- `fob_value_usd` — FOB trade value in USD
- `is_reported` — Whether data was directly reported (vs. estimated)

**Next:** `02_data_cleaning.ipynb` — compute China share, pre/post National Sword flags, reshape for analysis.